In [1]:
import kagglehub
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

In [2]:
K_PATH = kagglehub.competition_download("sem-eval-2026-task-13-subtask-a")

In [3]:
FEATURE_NAMES = [
    # Base features
    'avg_line_len', 
    'std_line_len', 
    'avg_indent', 
    'std_indent', 
    'max_indent',
    'empty_gap_std',
    'empty_gap_mean',
    
    # Comment features
    'n_single_comments',
    'text_comment_ratio',
]

In [4]:
tqdm.pandas()

In [5]:
import os
from utils import parallel_extract

TRAIN_PREPROCESS_PATH = 'train_preprocess.parquet'
TEST_PREPROCESS_PATH = 'test_preprocess.parquet'
VAL_PREPROCESS_PATH = 'val_preprocess.parquet'
TEST_SAMPLE_PREPROCESS_PATH = 'test_sample_preprocess.parquet'
RAW_TRAIN_PATH = os.path.join(K_PATH, 'Task_A', 'train.parquet')
RAW_TEST_PATH = os.path.join(K_PATH, 'Task_A', 'test.parquet')
RAW_TEST_SAMPLE_PATH = os.path.join(K_PATH, 'Task_A', 'test_sample.parquet')
RAW_VAL_PATH = os.path.join(K_PATH, 'Task_A', 'validation.parquet')

    
print('Reading train_df and test_df from parquet...')
train_df = pd.read_parquet(RAW_TRAIN_PATH)
test_df = pd.read_parquet(RAW_TEST_PATH)
val_df = pd.read_parquet(RAW_VAL_PATH)
test_sample_df = pd.read_parquet(RAW_TEST_SAMPLE_PATH)




if os.path.exists(TRAIN_PREPROCESS_PATH) and os.path.exists(TEST_PREPROCESS_PATH) and os.path.exists(VAL_PREPROCESS_PATH) and os.path.exists(TEST_SAMPLE_PREPROCESS_PATH):
    print(f'Already pre-extracted features from {TRAIN_PREPROCESS_PATH} and {TEST_PREPROCESS_PATH}...')
    
else:
    print(f'Extracting {len(FEATURE_NAMES)} handcrafted features in parallel...')

    # Chạy song song thay vì dùng list comprehension
    code_train_list = train_df['code'].to_list()
    X_train = parallel_extract(code_train_list, FEATURE_NAMES)
    y_train = train_df['label'].to_numpy()
    
    code_test_list = test_df['code'].to_list()
    X_test = parallel_extract(code_test_list, FEATURE_NAMES)
    
    code_val_list = val_df['code'].to_list()
    X_val = parallel_extract(code_val_list, FEATURE_NAMES)
    y_val = val_df['label'].to_numpy()
    
    code_test_sample_list = test_sample_df['code'].to_list()
    X_test_sample = parallel_extract(code_test_sample_list, FEATURE_NAMES)
    y_test_sample = test_sample_df['label'].to_numpy()

    # 1. Chuyển Numpy Array thành DataFrame
    train_features_df = pd.DataFrame(X_train, columns=FEATURE_NAMES)
    test_features_df = pd.DataFrame(X_test, columns=FEATURE_NAMES)
    val_features_df = pd.DataFrame(X_val, columns=FEATURE_NAMES)
    test_sample_features_df = pd.DataFrame(X_test_sample, columns=FEATURE_NAMES)

    # 2. Reset index của DF gốc để đảm bảo nối đúng dòng (rất quan trọng)
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_sample_df = test_sample_df.reset_index(drop=True)

    # 3. Nối ngang (axis=1) DataFrame gốc với DataFrame đặc trưng
    train_df = pd.concat([train_df, train_features_df], axis=1)
    test_df = pd.concat([test_df, test_features_df], axis=1)
    val_df = pd.concat([val_df, val_features_df], axis=1)
    test_sample_df = pd.concat([test_sample_df, test_sample_features_df], axis=1)

    print(f"Kích thước mới của dữ liệu: train={train_df.shape}, test={test_df.shape}, val={val_df.shape}, test_sample={test_sample_df.shape}")

    print(f'Saving full combined data to {TRAIN_PREPROCESS_PATH}, {TEST_PREPROCESS_PATH}, {VAL_PREPROCESS_PATH} and {TEST_SAMPLE_PREPROCESS_PATH}...')
    train_df.to_parquet(TRAIN_PREPROCESS_PATH, index=False)
    test_df.to_parquet(TEST_PREPROCESS_PATH, index=False)
    val_df.to_parquet(VAL_PREPROCESS_PATH, index=False)
    test_sample_df.to_parquet(TEST_SAMPLE_PREPROCESS_PATH, index=False)


Reading train_df and test_df from parquet...
Already pre-extracted features from train_preprocess.parquet and test_preprocess.parquet...


In [6]:

def lgb_macro_f1(y_true, y_pred):
    y_pred_bin = (y_pred > 0.5).astype(int)
    f1 = f1_score(y_true, y_pred_bin, average='macro')
    return 'macro_f1', f1, True

def train_lightgbm(train, val, test, test_sample, features, lgb_params):
    """
    Huấn luyện 1 mô hình LightGBM duy nhất trên toàn bộ tập train.
    Dùng tập val để đánh giá Early Stopping.
    """
    print("LightGBM training started")
    
    # 1. Trích xuất features
    X_train = train[features]
    y_train = train['label']
    
    X_val = val[features]  
    y_val = val['label']
    
    X_test = test[features] 
    X_test_sample = test_sample[features]
    
    # Khởi tạo mảng lưu kết quả dự đoán
    # Thay oof_preds thành train_preds để bạn có thể kiểm tra xem mô hình có bị overfit trên chính nó không
    train_preds = np.zeros(len(train))
    val_preds = np.zeros(len(val))
    test_preds = np.zeros(len(test))
    test_sample_preds = np.zeros(len(test_sample))
    # 2. Khởi tạo và fit mô hình
    model = lgb.LGBMClassifier(**lgb_params)
    
    print("\nĐang fit mô hình... (Theo dõi Macro F1 trên tập Validation)")
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=lgb_macro_f1, # Hàm custom của bạn
        # Set verbose=True hoặc 10 để xem tiến trình học và điểm số thay đổi ra sao
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)] 
    )
    
    # 3. Dự đoán xác suất lớp Positive (class 1)
    print("\nĐang tiến hành dự đoán...")
    train_preds = model.predict_proba(X_train)[:, 1]
    val_preds = model.predict_proba(X_val)[:, 1]
    test_preds = model.predict_proba(X_test)[:, 1]
    test_sample_preds = model.predict_proba(X_test_sample)[:, 1]
        
    print("\nQuá trình huấn luyện và dự đoán hoàn tất!")
    
    # Trả về model trực tiếp thay vì list
    return model, train_preds, val_preds, test_preds, test_sample_preds

# ==========================================
# CÁCH SỬ DỤNG
# ==========================================

# Parameters specifically chosen to prevent OOD memorization (Giữ nguyên)
lgb_params = {
    'objective': 'binary',
    'learning_rate': 0.05,
    'max_depth': 3,
    'num_leaves': 31,
    'colsample_bytree': 0.7, 
    'subsample': 0.8,
    'random_state': 42,
    'n_estimators': 500,
    'verbose': -1
}

train_pre_df = pd.read_parquet(TRAIN_PREPROCESS_PATH)[:350000]
test_pre_df = pd.read_parquet(TEST_PREPROCESS_PATH)
val_pre_df = pd.read_parquet(VAL_PREPROCESS_PATH)
test_sample_df = pd.read_parquet(TEST_SAMPLE_PREPROCESS_PATH)


# print(train_pre_df.columns)
models, oof_preds, val_preds, test_preds, test_sample_preds = train_lightgbm(train_pre_df, val_pre_df, test_pre_df, test_sample_df, FEATURE_NAMES, lgb_params)

LightGBM training started

Đang fit mô hình... (Theo dõi Macro F1 trên tập Validation)
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's binary_logloss: 0.171587	valid_0's macro_f1: 0.937599

Đang tiến hành dự đoán...

Quá trình huấn luyện và dự đoán hoàn tất!


In [7]:
import matplotlib.pyplot as plt

print("Best F1 score in validation set:")

thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_val = []
f1_scores_test_sample = []

for thresh in thresholds:
    # Use the validation set to find the best threshold
    preds_bin_val = (val_preds > thresh).astype(int)
    score = f1_score(val_pre_df['label'], preds_bin_val, average='macro')
    f1_scores_val.append(score)
    
    preds_bin_test_sample = (test_sample_preds > thresh).astype(int)
    score_test_sample = f1_score(test_sample_df['label'], preds_bin_test_sample, average='macro')
    f1_scores_test_sample.append(score_test_sample)

best_idx_val = np.argmax(f1_scores_val)
best_thresh_val = thresholds[best_idx_val]
best_f1_val = f1_scores_val[best_idx_val]

print(f"Best Validation Macro F1: {best_f1_val:.4f} at Threshold: {best_thresh_val:.2f}")

best_idx_test_sample = np.argmax(f1_scores_test_sample)
best_thresh_test_sample = thresholds[best_idx_test_sample]
best_f1_test_sample = f1_scores_test_sample[best_idx_test_sample]
print(f"Best Test Sample Macro F1: {best_f1_test_sample:.4f} at Threshold: {best_thresh_test_sample:.2f}")


Best F1 score in validation set:
Best Validation Macro F1: 0.9377 at Threshold: 0.57
Best Test Sample Macro F1: 0.6554 at Threshold: 0.89


In [8]:
threshold = 0.95

submit_preds = (test_preds > threshold).astype(int)

submission = pd.DataFrame({
    'ID': test_pre_df['ID'],
    'label': submit_preds
})

submission.to_csv('submission.csv', index=False)

print("Distribution of predictions in submission:")
print(submission['label'].value_counts(normalize=True) * 100)

Distribution of predictions in submission:
label
0    72.5254
1    27.4746
Name: proportion, dtype: float64


In [9]:
import subprocess
filename = 'submission.csv'
# Định nghĩa lệnh cần chạy dưới dạng một list các chuỗi
command = [
    "kaggle", 
    "competitions", 
    "submit", 
    "-c", "sem-eval-2026-task-13-subtask-a", 
    "-f", f"{filename}", 
    "-m", filename.replace('.csv', '')
]

try:
    # Chạy lệnh
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    
    # In ra kết quả thành công từ Kaggle
    print("Nộp bài thành công!")
    print(result.stdout)
    
except subprocess.CalledProcessError as e:
    # Bắt lỗi nếu lệnh CLI thất bại (ví dụ: sai tên file, chưa authenticate)
        print("Có lỗi xảy ra khi nộp bài:")
        print(e.stderr)

Nộp bài thành công!
Successfully submitted to SemEval-2026-Task13-Subtask-A
